In [1]:
# ============================================================
# EXPERIMENT 008
# XGBoost–CatBoost out-of-fold rank blend
# ============================================================

from pathlib import Path
from time import time
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier

try:
    from catboost import CatBoostClassifier
except ImportError as exc:
    raise ImportError(
        "CatBoost is not installed. Run `%pip install catboost`, "
        "restart the kernel if needed, and rerun the notebook."
    ) from exc


warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

EXPERIMENT_ID = "EXP-008"

RANDOM_STATE = 42
N_SPLITS = 3

TARGET = "addicted_label"
ID_COLUMN = "id"

HISTORICAL_EXP_007_OOF = 0.963969
MINIMUM_IMPROVEMENT = 0.0002


PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent


DATA_DIR = PROJECT_DIR / "data"
SUBMISSION_DIR = PROJECT_DIR / "submissions"
PREDICTION_DIR = PROJECT_DIR / "predictions"

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)


TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"


# ============================================================
# 2. LOAD DATA
# ============================================================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)


print(f"Train shape:             {train.shape}")
print(f"Test shape:              {test.shape}")
print(f"Sample submission shape: {sample_submission.shape}")


assert TARGET in train.columns
assert TARGET not in test.columns
assert len(test) == len(sample_submission)


y = train[TARGET].astype(int).copy()


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def safe_divide(
    numerator: pd.Series,
    denominator: pd.Series
) -> pd.Series:
    """
    Divide two Series and convert invalid results to missing.
    """

    safe_denominator = denominator.replace(0, np.nan)

    result = numerator / safe_denominator

    return result.replace(
        [np.inf, -np.inf],
        np.nan
    )


def add_behavioral_features(
    dataframe: pd.DataFrame
) -> pd.DataFrame:
    """
    Reproduce the behavioural features from EXP-007.
    """

    dataframe = dataframe.copy()

    dataframe["leisure_screen_time"] = (
        dataframe["social_media_hours"]
        + dataframe["gaming_hours"]
    )

    dataframe["leisure_share_of_screen"] = safe_divide(
        dataframe["leisure_screen_time"],
        dataframe["daily_screen_time_hours"]
    )

    dataframe["social_media_share_of_screen"] = safe_divide(
        dataframe["social_media_hours"],
        dataframe["daily_screen_time_hours"]
    )

    dataframe["gaming_share_of_screen"] = safe_divide(
        dataframe["gaming_hours"],
        dataframe["daily_screen_time_hours"]
    )

    dataframe["weekend_screen_change"] = (
        dataframe["weekend_screen_time"]
        - dataframe["daily_screen_time_hours"]
    )

    dataframe["weekend_to_daily_screen_ratio"] = safe_divide(
        dataframe["weekend_screen_time"],
        dataframe["daily_screen_time_hours"]
    )

    dataframe["screen_sleep_gap"] = (
        dataframe["daily_screen_time_hours"]
        - dataframe["sleep_hours"]
    )

    dataframe["screen_to_sleep_ratio"] = safe_divide(
        dataframe["daily_screen_time_hours"],
        dataframe["sleep_hours"]
    )

    dataframe["screen_to_work_ratio"] = safe_divide(
        dataframe["daily_screen_time_hours"],
        dataframe["work_study_hours"]
    )

    dataframe["leisure_to_work_ratio"] = safe_divide(
        dataframe["leisure_screen_time"],
        dataframe["work_study_hours"]
    )

    dataframe["notifications_per_screen_hour"] = safe_divide(
        dataframe["notifications_per_day"],
        dataframe["daily_screen_time_hours"]
    )

    dataframe["app_opens_per_screen_hour"] = safe_divide(
        dataframe["app_opens_per_day"],
        dataframe["daily_screen_time_hours"]
    )

    dataframe["notifications_per_app_open"] = safe_divide(
        dataframe["notifications_per_day"],
        dataframe["app_opens_per_day"]
    )

    return dataframe


def percentile_rank(values: np.ndarray) -> np.ndarray:
    """
    Convert predictions to percentile ranks between zero and one.
    """

    return (
        pd.Series(values)
        .rank(method="average", pct=True)
        .to_numpy()
    )


# ============================================================
# 4. PREPARE XGBOOST FEATURES
# ============================================================

X_xgb = train.drop(
    columns=[TARGET, ID_COLUMN]
).copy()

X_test_xgb = test.drop(
    columns=[ID_COLUMN]
).copy()


original_xgb_features = X_xgb.columns.tolist()


columns_with_missing_values = [
    column
    for column in original_xgb_features
    if (
        X_xgb[column].isna().any()
        or X_test_xgb[column].isna().any()
    )
]


for column in columns_with_missing_values:

    indicator_name = f"{column}__missing"

    X_xgb[indicator_name] = (
        X_xgb[column]
        .isna()
        .astype("int8")
    )

    X_test_xgb[indicator_name] = (
        X_test_xgb[column]
        .isna()
        .astype("int8")
    )


X_xgb = add_behavioral_features(X_xgb)
X_test_xgb = add_behavioral_features(X_test_xgb)


assert list(X_xgb.columns) == list(X_test_xgb.columns)


xgb_categorical_columns = X_xgb.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

xgb_numeric_columns = X_xgb.columns.difference(
    xgb_categorical_columns
).tolist()


print(f"\nXGBoost model features: {X_xgb.shape[1]}")


# ============================================================
# 5. PREPARE CATBOOST FEATURES
# ============================================================

# Keep CatBoost consistent with EXP-003 so it remains
# meaningfully different from the engineered XGBoost model.

X_cat = train.drop(
    columns=[TARGET, ID_COLUMN]
).copy()

X_test_cat = test.drop(
    columns=[ID_COLUMN]
).copy()


cat_categorical_columns = X_cat.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()


for column in cat_categorical_columns:

    X_cat[column] = (
        X_cat[column]
        .fillna("__MISSING__")
        .astype(str)
    )

    X_test_cat[column] = (
        X_test_cat[column]
        .fillna("__MISSING__")
        .astype(str)
    )


assert list(X_cat.columns) == list(X_test_cat.columns)


print(f"CatBoost model features: {X_cat.shape[1]}")


# ============================================================
# 6. XGBOOST PREPROCESSING
# ============================================================

try:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10,
        sparse_output=True
    )

except TypeError:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10,
        sparse=True
    )


numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)


categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "one_hot",
            one_hot_encoder
        )
    ]
)


xgb_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            xgb_numeric_columns
        ),
        (
            "categorical",
            categorical_pipeline,
            xgb_categorical_columns
        )
    ],
    remainder="drop"
)


# ============================================================
# 7. MODEL PARAMETERS
# ============================================================

# Identical to EXP-007.
xgb_parameters = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "n_estimators": 3000,
    "learning_rate": 0.05,
    "max_depth": 6,
    "min_child_weight": 5,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "early_stopping_rounds": 100,
    "random_state": RANDOM_STATE,
    "n_jobs": -1
}


# Identical to EXP-003.
catboost_parameters = {
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "iterations": 2000,
    "learning_rate": 0.05,
    "depth": 8,
    "l2_leaf_reg": 5.0,
    "random_strength": 1.0,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.80,
    "early_stopping_rounds": 100,
    "random_seed": RANDOM_STATE,
    "thread_count": -1,
    "task_type": "CPU",
    "allow_writing_files": False,
    "verbose": False
}


# ============================================================
# 8. CROSS-VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)


xgb_oof = np.zeros(len(train), dtype=float)
cat_oof = np.zeros(len(train), dtype=float)

xgb_test_predictions = np.zeros(len(test), dtype=float)
cat_test_predictions = np.zeros(len(test), dtype=float)


xgb_fold_scores = []
cat_fold_scores = []

xgb_best_iterations = []
cat_best_iterations = []


experiment_start = time()


for fold_number, (
    train_indices,
    validation_indices
) in enumerate(
    cv.split(X_xgb, y),
    start=1
):

    fold_start = time()

    y_train_fold = y.iloc[train_indices]
    y_validation_fold = y.iloc[validation_indices]


    # --------------------------------------------------------
    # XGBOOST
    # --------------------------------------------------------

    X_xgb_train_fold = X_xgb.iloc[train_indices]
    X_xgb_validation_fold = X_xgb.iloc[validation_indices]

    fold_preprocessor = clone(xgb_preprocessor)

    X_xgb_train_processed = (
        fold_preprocessor.fit_transform(
            X_xgb_train_fold
        )
    )

    X_xgb_validation_processed = (
        fold_preprocessor.transform(
            X_xgb_validation_fold
        )
    )

    X_xgb_test_processed = (
        fold_preprocessor.transform(
            X_test_xgb
        )
    )


    xgb_model = XGBClassifier(
        **xgb_parameters
    )

    xgb_model.fit(
        X_xgb_train_processed,
        y_train_fold,
        eval_set=[
            (
                X_xgb_validation_processed,
                y_validation_fold
            )
        ],
        verbose=False
    )


    xgb_validation_probabilities = (
        xgb_model.predict_proba(
            X_xgb_validation_processed
        )[:, 1]
    )

    xgb_fold_test_probabilities = (
        xgb_model.predict_proba(
            X_xgb_test_processed
        )[:, 1]
    )


    xgb_oof[validation_indices] = (
        xgb_validation_probabilities
    )

    xgb_test_predictions += (
        xgb_fold_test_probabilities / N_SPLITS
    )


    xgb_fold_auc = roc_auc_score(
        y_validation_fold,
        xgb_validation_probabilities
    )

    xgb_fold_scores.append(float(xgb_fold_auc))

    xgb_best_iterations.append(
        getattr(
            xgb_model,
            "best_iteration",
            None
        )
    )


    # --------------------------------------------------------
    # CATBOOST
    # --------------------------------------------------------

    X_cat_train_fold = X_cat.iloc[train_indices]
    X_cat_validation_fold = X_cat.iloc[validation_indices]


    cat_model = CatBoostClassifier(
        **catboost_parameters
    )

    cat_model.fit(
        X_cat_train_fold,
        y_train_fold,
        cat_features=cat_categorical_columns,
        eval_set=(
            X_cat_validation_fold,
            y_validation_fold
        ),
        use_best_model=True,
        verbose=False
    )


    cat_validation_probabilities = (
        cat_model.predict_proba(
            X_cat_validation_fold
        )[:, 1]
    )

    cat_fold_test_probabilities = (
        cat_model.predict_proba(
            X_test_cat
        )[:, 1]
    )


    cat_oof[validation_indices] = (
        cat_validation_probabilities
    )

    cat_test_predictions += (
        cat_fold_test_probabilities / N_SPLITS
    )


    cat_fold_auc = roc_auc_score(
        y_validation_fold,
        cat_validation_probabilities
    )

    cat_fold_scores.append(float(cat_fold_auc))

    cat_best_iterations.append(
        cat_model.get_best_iteration()
    )


    fold_minutes = (
        time() - fold_start
    ) / 60


    print(
        f"Fold {fold_number}/{N_SPLITS} | "
        f"XGB AUC: {xgb_fold_auc:.6f} | "
        f"CatBoost AUC: {cat_fold_auc:.6f} | "
        f"Time: {fold_minutes:.2f} minutes"
    )


# ============================================================
# 9. INDIVIDUAL MODEL RESULTS
# ============================================================

xgb_oof_auc = roc_auc_score(
    y,
    xgb_oof
)

cat_oof_auc = roc_auc_score(
    y,
    cat_oof
)


prediction_correlation = np.corrcoef(
    xgb_oof,
    cat_oof
)[0, 1]


print("\n" + "=" * 60)
print("INDIVIDUAL MODEL RESULTS")
print("=" * 60)

print(f"XGBoost OOF AUC:           {xgb_oof_auc:.6f}")
print(f"Historical EXP-007 OOF:    {HISTORICAL_EXP_007_OOF:.6f}")
print(f"CatBoost OOF AUC:          {cat_oof_auc:.6f}")
print(f"OOF prediction correlation:{prediction_correlation: .6f}")

print(f"\nXGBoost fold scores: {xgb_fold_scores}")
print(f"CatBoost fold scores: {cat_fold_scores}")

print(f"\nXGBoost best iterations: {xgb_best_iterations}")
print(f"CatBoost best iterations: {cat_best_iterations}")


# ============================================================
# 10. SAVE COMPONENT PREDICTIONS
# ============================================================

oof_components = pd.DataFrame(
    {
        ID_COLUMN: train[ID_COLUMN],
        TARGET: y,
        "xgb_oof_probability": xgb_oof,
        "catboost_oof_probability": cat_oof
    }
)


test_components = pd.DataFrame(
    {
        ID_COLUMN: test[ID_COLUMN],
        "xgb_test_probability": xgb_test_predictions,
        "catboost_test_probability": cat_test_predictions
    }
)


oof_components_path = (
    PREDICTION_DIR
    / "exp_008_oof_components.csv"
)

test_components_path = (
    PREDICTION_DIR
    / "exp_008_test_components.csv"
)


oof_components.to_csv(
    oof_components_path,
    index=False
)

test_components.to_csv(
    test_components_path,
    index=False
)


print(f"\nSaved OOF components to:\n{oof_components_path}")
print(f"\nSaved test components to:\n{test_components_path}")


# ============================================================
# 11. RANK TRANSFORMATION
# ============================================================

xgb_oof_rank = percentile_rank(xgb_oof)
cat_oof_rank = percentile_rank(cat_oof)

xgb_test_rank = percentile_rank(
    xgb_test_predictions
)

cat_test_rank = percentile_rank(
    cat_test_predictions
)


# ============================================================
# 12. TEST BLEND WEIGHTS
# ============================================================

# Test a small predefined grid.
# Weight refers to the XGBoost contribution.

xgb_weights = np.arange(
    0.50,
    1.001,
    0.05
)


blend_results = []


for xgb_weight in xgb_weights:

    catboost_weight = 1.0 - xgb_weight

    blended_oof_rank = (
        xgb_weight * xgb_oof_rank
        + catboost_weight * cat_oof_rank
    )

    blend_auc = roc_auc_score(
        y,
        blended_oof_rank
    )

    blend_results.append(
        {
            "xgb_weight": float(xgb_weight),
            "catboost_weight": float(catboost_weight),
            "oof_auc": float(blend_auc),
            "gain_vs_xgb": float(
                blend_auc - xgb_oof_auc
            )
        }
    )


blend_results = pd.DataFrame(
    blend_results
).sort_values(
    "oof_auc",
    ascending=False
).reset_index(drop=True)


print("\n" + "=" * 60)
print("RANK BLEND RESULTS")
print("=" * 60)

print(
    blend_results.round(6)
)


best_xgb_weight = float(
    blend_results.loc[0, "xgb_weight"]
)

best_catboost_weight = float(
    blend_results.loc[0, "catboost_weight"]
)

best_blend_oof_auc = float(
    blend_results.loc[0, "oof_auc"]
)

blend_gain_vs_xgb = (
    best_blend_oof_auc - xgb_oof_auc
)


print("\nBest blend:")

print(
    f"XGBoost weight:     "
    f"{best_xgb_weight:.2f}"
)

print(
    f"CatBoost weight:    "
    f"{best_catboost_weight:.2f}"
)

print(
    f"Blend OOF AUC:      "
    f"{best_blend_oof_auc:.6f}"
)

print(
    f"Gain vs XGBoost:    "
    f"{blend_gain_vs_xgb:+.6f}"
)


# ============================================================
# 13. INTERPRET RESULT
# ============================================================

print("\nInterpretation:")


if blend_gain_vs_xgb >= MINIMUM_IMPROVEMENT:

    print(
        "The rank blend produced a useful improvement. "
        "CatBoost contributes complementary ranking information."
    )

elif blend_gain_vs_xgb > -MINIMUM_IMPROVEMENT:

    print(
        "The blend is effectively unchanged. CatBoost does not "
        "add enough complementary information to justify ensembling."
    )

else:

    print(
        "The blend reduced performance. Retain XGBoost alone."
    )


# ============================================================
# 14. BUILD BEST TEST BLEND
# ============================================================

best_blended_test_predictions = (
    best_xgb_weight * xgb_test_rank
    + best_catboost_weight * cat_test_rank
)


assert np.isfinite(
    best_blended_test_predictions
).all()

assert (
    (best_blended_test_predictions >= 0)
    & (best_blended_test_predictions <= 1)
).all()


print("\nBlended test prediction summary:")

print(
    pd.Series(
        best_blended_test_predictions,
        name="rank_blend"
    ).describe()
)


# ============================================================
# 15. CONDITIONAL SUBMISSION
# ============================================================

if blend_gain_vs_xgb >= MINIMUM_IMPROVEMENT:

    submission = sample_submission.copy()

    assert TARGET in submission.columns

    submission[TARGET] = (
        best_blended_test_predictions
    )


    submission_path = (
        SUBMISSION_DIR
        / "exp_008_xgb_catboost_rank_blend.csv"
    )


    submission.to_csv(
        submission_path,
        index=False
    )


    print(
        "\nSubmission created because the blend improved "
        "XGBoost OOF by at least "
        f"{MINIMUM_IMPROVEMENT:.4f}."
    )

    print(f"Saved to:\n{submission_path}")

    print("\nSubmission preview:")
    print(submission.head())


else:

    submission_path = None

    print(
        "\nNo submission created because the blend did not "
        "provide a sufficiently large OOF improvement."
    )


# ============================================================
# 16. RUNTIME
# ============================================================

total_minutes = (
    time() - experiment_start
) / 60


print(
    f"\nTotal runtime: "
    f"{total_minutes:.2f} minutes"
)

Train shape:             (691369, 14)
Test shape:              (296302, 13)
Sample submission shape: (296302, 2)

XGBoost model features: 37
CatBoost model features: 12
Fold 1/3 | XGB AUC: 0.963396 | CatBoost AUC: 0.961586 | Time: 6.10 minutes
Fold 2/3 | XGB AUC: 0.964245 | CatBoost AUC: 0.962336 | Time: 6.31 minutes
Fold 3/3 | XGB AUC: 0.964271 | CatBoost AUC: 0.962426 | Time: 6.53 minutes

INDIVIDUAL MODEL RESULTS
XGBoost OOF AUC:           0.963969
Historical EXP-007 OOF:    0.963969
CatBoost OOF AUC:          0.962115
OOF prediction correlation: 0.988373

XGBoost fold scores: [0.9633957267702942, 0.9642452590852241, 0.964270842436306]
CatBoost fold scores: [0.9615863694416876, 0.9623356996971737, 0.9624261478327517]

XGBoost best iterations: [2369, 2312, 2194]
CatBoost best iterations: [1999, 1999, 1998]

Saved OOF components to:
C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\02-smartphone-addiction\predictions\exp_008_oof_components.csv

Saved test components to:
C

### Experiment Log

In [2]:
from datetime import datetime
from pathlib import Path
import json

import numpy as np
import pandas as pd


EXPERIMENT_LOG_PATH = PROJECT_DIR / "experiment_log.csv"


def log_experiment(
    experiment_id,
    description,
    model,
    features,
    validation_method,
    cv_scores,
    kaggle_score=None,
    changes="",
    submission_file="",
    notes="",
    log_path=EXPERIMENT_LOG_PATH
):
    """
    Add or update one experiment in experiment_log.csv.

    If the experiment_id already exists, its previous row is replaced.
    """

    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    cv_scores = [float(score) for score in cv_scores]

    cv_mean = float(np.mean(cv_scores))
    cv_std = float(np.std(cv_scores))

    kaggle_score_value = (
        float(kaggle_score)
        if kaggle_score is not None
        else np.nan
    )

    kaggle_cv_gap = (
        kaggle_score_value - cv_mean
        if pd.notna(kaggle_score_value)
        else np.nan
    )

    experiment_record = {
        "experiment_id": experiment_id,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "description": description,
        "model": model,
        "features": json.dumps(list(features)),
        "n_features": len(features),
        "validation_method": validation_method,
        "cv_scores": json.dumps(cv_scores),
        "cv_mean": cv_mean,
        "cv_std": cv_std,
        "kaggle_score": kaggle_score_value,
        "kaggle_cv_gap": kaggle_cv_gap,
        "changes": changes,
        "submission_file": submission_file,
        "notes": notes
    }

    if log_path.exists():
        experiments = pd.read_csv(log_path)

        # Prevent duplicate rows when rerunning the same experiment cell.
        if "experiment_id" in experiments.columns:
            experiments = experiments[
                experiments["experiment_id"] != experiment_id
            ].copy()
    else:
        experiments = pd.DataFrame()

    new_row = pd.DataFrame([experiment_record])

    experiments = pd.concat(
        [experiments, new_row],
        ignore_index=True
    )

    experiments = experiments.sort_values(
        by="experiment_id"
    ).reset_index(drop=True)

    experiments.to_csv(log_path, index=False)

    print(f"Logged {experiment_id}")
    print(f"CV mean:       {cv_mean:.6f}")
    print(f"CV SD:         {cv_std:.6f}")

    if pd.notna(kaggle_score_value):
        print(f"Kaggle score:  {kaggle_score_value:.6f}")
        print(f"Kaggle-CV gap: {kaggle_cv_gap:+.6f}")

    print(f"Log saved to:  {log_path}")

    return experiments

In [3]:
experiments = log_experiment(
    experiment_id="EXP-008",
    description=(
        "Out-of-fold percentile-rank blend of the EXP-007 engineered "
        "XGBoost model and the EXP-003 CatBoost model."
    ),
    model="XGBoost + CatBoost rank blend",
    features=X_xgb.columns.tolist(),
    validation_method="3-fold StratifiedKFold with ROC AUC",
    cv_scores=[
        0.9633957267702942,
        0.9642452590852241,
        0.9642708424363060
    ],
    kaggle_score=None,
    changes=(
        "Retrained XGBoost and CatBoost on identical folds, converted "
        "their OOF and test probabilities to percentile ranks, and tested "
        "XGBoost blend weights from 0.50 through 1.00."
    ),
    submission_file="",
    notes=(
        "XGBoost OOF AUC was 0.963969 and CatBoost OOF AUC was 0.962115. "
        "OOF prediction correlation was 0.988373. The best blend used "
        "75% XGBoost and 25% CatBoost, achieving OOF AUC 0.964144, a gain "
        "of only 0.000176. This failed the predefined 0.0002 improvement "
        "threshold, so no Kaggle submission was created. EXP-007 remains "
        "the final selected model."
    )
)

experiments.tail()

Logged EXP-008
CV mean:       0.963971
CV SD:         0.000407
Log saved to:  C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\02-smartphone-addiction\experiment_log.csv


,experiment_id,timestamp,description,model,features,n_features,validation_method,cv_scores,cv_mean,cv_std,kaggle_score,kaggle_cv_gap,changes,submission_file,notes
3,EXP-004,2026-08-02 22:18:27,XGBoost baseline extended with binary missingn...,XGBClassifier,"[""age"", ""daily_screen_time_hours"", ""social_med...",24,3-fold StratifiedKFold with ROC AUC,"[0.962723, 0.963675, 0.963473]",0.963290,0.000410,0.96475,0.001460,Added one binary missingness indicator for eac...,exp_004_xgb_missing_indicators.csv,OOF AUC was 0.963290 with fold SD 0.000409. Th...
4,EXP-005,2026-08-02 22:27:47,XGBoost missingness-indicator model with the e...,XGBClassifier,"[""age"", ""daily_screen_time_hours"", ""social_med...",24,3-fold StratifiedKFold with ROC AUC,"[0.963096, 0.963999, 0.963887]",0.963661,0.000402,0.96534,0.001679,"Retained all EXP-004 features, missingness ind...",exp_005_xgb_more_trees.csv,"OOF AUC was 0.963659 with fold SD 0.000402, im..."
5,EXP-006,2026-08-02 22:43:36,XGBoost missingness-indicator model using a lo...,XGBClassifier,"[""age"", ""daily_screen_time_hours"", ""social_med...",24,3-fold StratifiedKFold with ROC AUC,"[0.963223, 0.964099, 0.964023]",0.963782,0.000396,NaN,NaN,"Retained the EXP-005 data, features, preproces...",NaN,"OOF AUC was 0.963778 with fold SD 0.000396, im..."
6,EXP-007,2026-08-02 22:54:47,"XGBoost model using original features, missing...",XGBClassifier,"[""age"", ""daily_screen_time_hours"", ""social_med...",37,3-fold StratifiedKFold with ROC AUC,"[0.963396, 0.964245, 0.964271]",0.963971,0.000406,0.96559,0.001619,Retained the EXP-005 model configuration and a...,exp_007_behavioral_features.csv,"OOF AUC was 0.963969 with fold SD 0.000407, im..."
7,EXP-008,2026-08-02 23:16:44,Out-of-fold percentile-rank blend of the EXP-0...,XGBoost + CatBoost rank blend,"[""age"", ""daily_screen_time_hours"", ""social_med...",37,3-fold StratifiedKFold with ROC AUC,"[0.9633957267702942, 0.9642452590852241, 0.964...",0.963971,0.000407,NaN,NaN,Retrained XGBoost and CatBoost on identical fo...,,XGBoost OOF AUC was 0.963969 and CatBoost OOF ...
